Supplementary code for the [Rearchitecting LLMs](https://www.manning.com/books/rearchitecting-llms) book by [Pere Martra](https://github.com/peremartra).
Code repository: [https://github.com/peremartra/Rearchitecting-LLMs](https://github.com/peremartra/Rearchitecting-LLMs)

# Rearchitecting LLMs
### Structural techniques for efficient models

Chapter 11: Fairness-Aware Pruning — NB02: BBQ Benchmark

[LinkedIn](https://www.linkedin.com/in/peremartra/) | [GitHub](https://github.com/peremartra) | [X](https://x.com/PereMartra) | [Hugging Face](https://huggingface.co/oopere)

---

Colab Environment: GPU T4 (also runs on CPU, only slower)

Model:
* `meta-llama/Llama-3.2-1B` — the model we instrument and intervene on

---

This notebook measures whether the neuron intervention from Chapter 11, section 11.5 (`EXPLICIT_SCALES`, the Top-5 neurons selected on the police-encounter prompt pair) has any measurable effect on a standard bias benchmark, rather than on the handful of hand-picked prompts used earlier in the chapter.

We use [BBQ](https://github.com/nyu-mll/BBQ) (Bias Benchmark for QA), specifically its `race_ethnicity` category, since that is the axis the intervention in 11.5 targets. BBQ presents a context, a question, and three answer choices (the stereotyped group, the other group, and an unknown option), in two variants: an ambiguous context, where the correct answer is always unknown, and a disambiguated context, where the correct answer can be determined. For each variant, BBQ reports a bias score, the rate at which the model's answer reflects the stereotype rather than the correct or unknown answer, and an accuracy score.

We run this benchmark twice on `meta-llama/Llama-3.2-1B`, once with the original weights and once with `EXPLICIT_SCALES` applied, and compare the two runs. This notebook does not repeat any of the activation-capture or SignedScore machinery from NB01. It only reloads the model, reapplies the same five-neuron intervention, and runs it through a standard benchmark.

## Setup

This section installs `lm-evaluation-harness`, the framework we use to run BBQ, and loads the model. Everything here is copied from NB01 without modification, since the point of this notebook is to reuse the same model and the same intervention, not to build new infrastructure.

In [ ]:
!pip install -q transformers torch lm_eval

In [ ]:
import copy
import random
from typing import Dict, Tuple

import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

Same seeding routine used across the book's notebooks whenever weight modification is involved.

In [ ]:
def set_seed(seed=42):
    """Set random seed for reproducibility."""
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)
print("Random seed set to 42")

In [ ]:
MODEL_ID = "meta-llama/Llama-3.2-1B"

# How many examples to sample per BBQ context (ambiguous / disambiguated) for a quick run.
# Set to None to run the full race_ethnicity subset (6,880 examples).
# Start with a small limit to measure the per-example time on T4 before committing to a full run.
BBQ_LIMIT = 100

print(f"MODEL_ID: {MODEL_ID}")
print(f"BBQ_LIMIT: {BBQ_LIMIT}")

In [ ]:
print(f"Loading tokenizer: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print(f"Loading model: {MODEL_ID}")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
)
model = model.to(DEVICE)
model.eval()

print("Model loaded.")
print(f"  Layers: {model.config.num_hidden_layers}")
print(f"  Hidden size: {model.config.hidden_size}")

# Kept in memory so we can restore the model between the baseline and intervened runs
# without reloading from disk each time (see NB01, section 11.5).
original_state_dict = copy.deepcopy(model.state_dict())

## Reapplying the race intervention

`get_model_layers`, `apply_explicit_scales`, and `restore_model` are unchanged from NB01, sections 11.1 and 11.5. `EXPLICIT_SCALES` is the same Top-5 neuron set selected in section 11.4 on the police-encounter prompt pair, scaled down by the same factor of 0.5. Nothing here is recomputed, we only reapply the intervention NB01 already validated on individual prompts.

In [ ]:
def get_model_layers(model):
    """Resolve the list of transformer decoder layers across common architectures."""
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        return model.model.layers
    if hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        return model.transformer.h
    raise ValueError("Could not resolve decoder layers for this model architecture.")


def apply_explicit_scales(model, scales: Dict[Tuple[int, int], float]):
    """
    Apply an explicit per-neuron scale factor to gate_proj (row), up_proj (row),
    and down_proj (column) for each (layer_idx, neuron_idx) key in `scales`.
    """
    layers = get_model_layers(model)
    with torch.no_grad():
        for (layer_idx, neuron_idx), scale in scales.items():
            mlp = layers[layer_idx].mlp
            mlp.gate_proj.weight.data[neuron_idx, :] *= scale
            mlp.up_proj.weight.data[neuron_idx, :] *= scale
            mlp.down_proj.weight.data[:, neuron_idx] *= scale
    return model


def restore_model():
    model.load_state_dict(original_state_dict)
    model.eval()

In [ ]:
# Race-intervention neurons, Top-5 by |SignedScore| on the police-encounter pair (NB01, section 11.4).
EXPLICIT_SCALES = {
    (15, 1788): 0.5,  # positive-difference
    (15, 1070): 0.5,  # negative-difference
    (15, 8109): 0.5,  # positive-difference
    (15, 2719): 0.5,  # positive-difference
    (15, 3024): 0.5,  # positive-difference
}

## BBQ evaluation

`run_bbq` wraps the already-loaded `model` and `tokenizer` in lm-evaluation-harness's `HFLM` adapter and runs the `bbq_race_ethnicity` task, zero-shot. We deliberately do not reuse this book's shared `model_evaluation` helper here. That helper only keeps a fixed list of metric names (accuracy, accuracy norm, perplexity, f1, exact match), and BBQ's own `acc` metric matches one of those names, so the helper stops there and silently drops the bias-score metrics we actually need. `run_bbq` reads the five metrics we care about directly from the raw results dictionary instead, and prints that raw dictionary once so the exact key names can be checked against whatever lm-eval version is installed.

In [ ]:
def run_bbq(model, tokenizer, limit=None, show_raw=False):
    """
    Evaluate the model on the BBQ race/ethnicity subset, zero-shot.

    Returns overall accuracy, accuracy in ambiguous and disambiguated context, and the
    corresponding bias scores. Set show_raw=True once to inspect the full metrics dictionary
    lm-eval returns, in case key names differ across lm-eval versions.
    """
    from lm_eval import evaluator
    from lm_eval.models.huggingface import HFLM

    model_wrapper = HFLM(pretrained=model, tokenizer=tokenizer, device=DEVICE)
    results = evaluator.simple_evaluate(
        model=model_wrapper,
        tasks=["bbq_race_ethnicity"],
        num_fewshot=0,
        limit=limit,
        device=DEVICE,
    )
    res = results["results"]["bbq_race_ethnicity"]

    if show_raw:
        print("Raw bbq_race_ethnicity metrics:")
        for key, value in res.items():
            print(f"  {key}: {value}")

    return {
        "acc": res["acc,none"],
        "accuracy_amb": res["accuracy_amb,none"],
        "accuracy_disamb": res["accuracy_disamb,none"],
        "amb_bias_score": res["amb_bias_score,none"],
        "disamb_bias_score": res["disamb_bias_score,none"],
    }

First the baseline. We restore the original weights, then run `run_bbq` with `show_raw=True` so the raw metric names are visible before we rely on them for the intervened run and the chart.

In [ ]:
restore_model()

baseline_bbq = run_bbq(model, tokenizer, limit=BBQ_LIMIT, show_raw=True)

print("\nBaseline BBQ (race/ethnicity):")
for key, value in baseline_bbq.items():
    print(f"  {key}: {value:+.4f}")

Now the same evaluation with `EXPLICIT_SCALES` applied. No recomputation of the intervention itself, the five neurons and the 0.5 scale factor are exactly the ones validated in NB01.

In [ ]:
apply_explicit_scales(model, EXPLICIT_SCALES)

race_bbq = run_bbq(model, tokenizer, limit=BBQ_LIMIT)

print("Intervened BBQ (race/ethnicity):")
for key, value in race_bbq.items():
    print(f"  {key}: {value:+.4f}")

## Comparing baseline and race intervention

Two panels, side by side, both broken down by context (ambiguous, disambiguated) so they line up visually. The left panel shows the bias score, the metric that most directly answers whether the intervention reduces stereotyped answers. The right panel shows accuracy for the same two contexts, as a check that the intervention is not simply degrading the model's ability to answer the question at all. Bar hatching, not color, carries the distinction between baseline and intervened, consistent with the book's black-and-white printing. Delta labels are absolute differences (intervened minus baseline), not percentages, since the bias-score baseline values are close to zero and a percentage change would be unstable there.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

contexts = ["Ambiguous", "Disambiguated"]
x = np.arange(len(contexts))
bar_width = 0.32

bias_baseline = [baseline_bbq["amb_bias_score"], baseline_bbq["disamb_bias_score"]]
bias_race     = [race_bbq["amb_bias_score"],     race_bbq["disamb_bias_score"]]
acc_baseline  = [baseline_bbq["accuracy_amb"],    baseline_bbq["accuracy_disamb"]]
acc_race      = [race_bbq["accuracy_amb"],        race_bbq["accuracy_disamb"]]

fig, (ax_bias, ax_acc) = plt.subplots(1, 2, figsize=(11, 5))


def plot_panel(ax, baseline_values, race_values, title, ylabel, symmetric):
    bars_baseline = ax.bar(
        x - bar_width / 2, baseline_values,
        width=bar_width,
        color="lightblue", edgecolor="black", linewidth=1.2, hatch="//",
        label="Baseline",
    )
    bars_race = ax.bar(
        x + bar_width / 2, race_values,
        width=bar_width,
        color="orange", edgecolor="black", linewidth=1.2, hatch="\\\\",
        label="Race intervention (EXPLICIT_SCALES)",
    )

    # y-limits are set before the labels so the label offset can be a fraction
    # of each panel's own range, instead of one fixed value that would only
    # make sense for one of the two very different scales (bias score vs accuracy).
    if symmetric:
        bound = max(0.05, max(abs(v) for v in baseline_values + race_values) * 1.4)
        ax.set_ylim(-bound, bound)
        ax.axhline(0, color="black", linewidth=0.8)
    else:
        ax.set_ylim(0, min(1.0, max(baseline_values + race_values) * 1.3))

    y_span = ax.get_ylim()[1] - ax.get_ylim()[0]
    label_offset = 0.04 * y_span

    for base_val, bar in zip(baseline_values, bars_race):
        delta = bar.get_height() - base_val
        going_up = bar.get_height() >= 0
        y_text = bar.get_height() + label_offset if going_up else bar.get_height() - label_offset
        ax.text(
            bar.get_x() + bar.get_width() / 2, y_text, f"{delta:+.3f}",
            ha="center", va="bottom" if going_up else "top",
            fontsize=9, color="#333333",
        )

    ax.set_xticks(x)
    ax.set_xticklabels(contexts, fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontsize=12, pad=10)
    ax.yaxis.grid(True, linestyle="--", alpha=0.6)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)


plot_panel(ax_bias, bias_baseline, bias_race, "BBQ bias score (race/ethnicity)", "Bias score", symmetric=True)
plot_panel(ax_acc, acc_baseline, acc_race, "BBQ accuracy (race/ethnicity)", "Accuracy", symmetric=False)

handles, labels = ax_bias.get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 1.04), ncol=2, fontsize=10)

plt.tight_layout()
plt.savefig("bbq_race_ethnicity_results.png", dpi=150, bbox_inches="tight")
plt.show()

TODO(Pere): once run on the full `race_ethnicity` subset (`BBQ_LIMIT = None`), add prose here comparing the two panels: does the bias score move toward zero after the intervention without a corresponding drop in accuracy, or does one come at the cost of the other? If the accuracy panel barely changes between baseline and intervened, consider dropping it and keeping only the bias-score panel, as discussed before writing this notebook.